# Evaluate a model

Run on the base model first, then on each trained model with the same settings (change `MODEL` and `RUN_NAME`). Use an L4 GPU. Results go to `results/<RUN_NAME>/`.

`eval/run_eval.py` does the same for all models in one go.

In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/sml-project"
DATA_DIR    = f"{PROJECT_DIR}/data/final_python"

MODEL    = "Qwen/Qwen3.5-0.8B"   # later: f"{PROJECT_DIR}/checkpoints/python_coder/merged"
RUN_NAME = "base"                # later: "python_coder", "python_selfcritique", ...

RUN_CODEGEN    = True
THINKING_MODES = [False, True]   # base: both. coder/judge: [False]. selfcritique: [True]
DATASETS       = ["humaneval", "mbpp"]
MAX_NEW_TOKENS = {False: 1024, True: 4096}

RUN_JUDGE      = True            # base model as judge (baseline); later the trained judge
JUDGE_SAMPLE   = 600             # judge test examples to score (fixed, seeded sample)

BATCH_SIZE = 16
SEED = 0
QUICK_CHECK = True               # first run: 5 problems only, to see the pipeline works

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Qwen3.5 needs a recent transformers
!pip install -q -U transformers accelerate peft evalplus
# the torchao on Colab breaks peft, we don't use it
!pip uninstall -y -q torchao
# optional, faster linear attention
!pip install -q flash-linear-attention || echo "flash-linear-attention not installed (optional)"

import sys, torch, transformers, peft
sys.path[:0] = [f"{PROJECT_DIR}/{d}" for d in ["data", "train", "eval"]]
print("transformers", transformers.__version__, "| peft", peft.__version__, "| torch", torch.__version__)
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > L4 GPU"
print("GPU:", torch.cuda.get_device_name(0), "| bf16:", torch.cuda.is_bf16_supported())

In [ ]:
from transformers import AutoTokenizer

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
tokenizer = AutoTokenizer.from_pretrained(MODEL)
import transformers as tf
# Qwen3.5-0.8B is multimodal, try the classes that can load it
model = None
for cls_name in ["AutoModelForMultimodalLM", "AutoModelForImageTextToText", "AutoModelForCausalLM"]:
    cls = getattr(tf, cls_name, None)
    if cls is None:
        continue
    try:
        model = cls.from_pretrained(MODEL, torch_dtype=dtype, device_map="cuda")
        print("loaded with", cls_name)
        break
    except (ValueError, KeyError) as err:
        print(cls_name, "failed:", str(err)[:120])
assert model is not None, "could not load the model: upgrade transformers (pip install -U transformers)"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(type(model).__name__, f"{sum(p.numel() for p in model.parameters()) / 1e6:.0f}M params")

In [ ]:
# check that enable_thinking changes the prompt
probe = [{"role": "user", "content": "hi"}]
off = tokenizer.apply_chat_template(probe, tokenize=False, add_generation_prompt=True, enable_thinking=False)
on = tokenizer.apply_chat_template(probe, tokenize=False, add_generation_prompt=True, enable_thinking=True)
print("thinking OFF ends with:", repr(off[-60:]))
print("thinking ON  ends with:", repr(on[-60:]))
assert off != on, ("enable_thinking has no effect on this chat template. Stop and check the "
                   "model card for how thinking is switched on/off before continuing.")

## Code: HumanEval+ and MBPP+

In [ ]:
import json, subprocess, glob, os, statistics
from pathlib import Path
import eval_utils as E
from evalplus.data import get_human_eval_plus, get_mbpp_plus
try:
    from evalplus.sanitize import sanitize
except ImportError:
    sanitize = None
    print("evalplus.sanitize not found: using plain code extraction")

RESULTS = Path(f"{PROJECT_DIR}/results/{RUN_NAME}")
RESULTS.mkdir(parents=True, exist_ok=True)
LOADERS = {"humaneval": get_human_eval_plus, "mbpp": get_mbpp_plus}
summary = {}

def run_codegen(dataset, thinking):
    problems = LOADERS[dataset]()
    task_ids = sorted(problems)[:5] if QUICK_CHECK else sorted(problems)
    convs = [E.codegen_messages(problems[t]["prompt"]) for t in task_ids]
    gens = E.generate(model, tokenizer, convs, enable_thinking=thinking,
                      max_new_tokens=MAX_NEW_TOKENS[thinking], batch_size=BATCH_SIZE, seed=SEED)
    tag = f"{dataset}_{'think' if thinking else 'nothink'}"
    samples, unfinished = [], 0
    for t, g in zip(task_ids, gens):
        answer, finished = E.strip_thinking(g["text"], thinking)
        unfinished += not finished
        code_text = E.extract_code(answer)
        if sanitize:
            try:
                code_text = sanitize(code_text, problems[t]["entry_point"])
            except Exception:
                pass
        samples.append({"task_id": t, "solution": code_text})
    sample_path = RESULTS / f"{tag}_samples.jsonl"
    full = E.fill_missing_problems(samples, sorted(problems))   # EvalPlus needs every problem
    sample_path.write_text("\n".join(json.dumps(s) for s in full))
    (RESULTS / f"{tag}_raw.jsonl").write_text("\n".join(
        json.dumps({"task_id": t, **g}) for t, g in zip(task_ids, gens)))

    for old in glob.glob(str(RESULTS / f"{tag}_samples*eval_results.json")) + glob.glob(str(RESULTS / "eval_results.json")):
        os.remove(old)                       # otherwise evalplus asks to reuse them
    subprocess.run(["evalplus.evaluate", "--dataset", dataset, "--samples", str(sample_path)],
                   check=True, input="Y\n", text=True)
    found = (sorted(glob.glob(str(RESULTS / f"{tag}_samples*eval_results.json")))
             or sorted(glob.glob(str(RESULTS / "eval_results.json"))))     # older evalplus name
    assert found, f"no EvalPlus results file in {RESULTS}"
    result_file = found[-1]
    per_task = E.parse_evalplus_results(result_file)
    per_task = {t: per_task[t] for t in task_ids if t in per_task}

    rate, lo, hi = E.bootstrap_ci([v["plus"] for v in per_task.values()], seed=SEED)
    stats = {
        "pass@1_plus": rate, "ci95": [lo, hi],
        "pass@1_base": sum(v["base"] for v in per_task.values()) / len(per_task),
        "problems": len(per_task), "thinking_unfinished": unfinished,
        "mean_new_tokens": statistics.mean(g["new_tokens"] for g in gens),
        "mean_seconds": statistics.mean(g["seconds"] for g in gens),
        "decoding": E.DECODING["thinking" if thinking else "greedy"],
        "max_new_tokens": MAX_NEW_TOKENS[thinking], "model": MODEL,
    }
    (RESULTS / f"{tag}_per_task.json").write_text(json.dumps(per_task, indent=1))
    (RESULTS / f"{tag}_summary.json").write_text(json.dumps(stats, indent=1))
    print(f"{tag}: pass@1+ {rate:.1%} [{lo:.1%}-{hi:.1%}]  base tests {stats['pass@1_base']:.1%}  "
          f"tokens {stats['mean_new_tokens']:.0f}  unfinished thinking {unfinished}")
    return stats

if RUN_CODEGEN:
    for dataset in DATASETS:
        for thinking in THINKING_MODES:
            summary[f"{dataset}_{'think' if thinking else 'nothink'}"] = run_codegen(dataset, thinking)

Check a few raw outputs before trusting the numbers.

In [ ]:
for path in sorted(RESULTS.glob("*_raw.jsonl"))[:2]:
    first = json.loads(path.read_text().splitlines()[0])
    print(f"=== {path.name} · {first['task_id']} · {first['new_tokens']} tokens ===")
    print(first["text"][:1500], "\n")

## Model as judge

In [ ]:
import random
import prompts as P
from sft_utils import load_jsonl

if RUN_JUDGE:
    judge_test = load_jsonl(f"{DATA_DIR}/judge_test.jsonl")
    rng = random.Random(SEED)
    sample = rng.sample(judge_test, min(JUDGE_SAMPLE, len(judge_test)))
    if QUICK_CHECK:
        sample = sample[:10]
    convs = [ex["messages"][:-1] for ex in sample]          # system + user, no answer
    gens = E.generate(model, tokenizer, convs, enable_thinking=False, max_new_tokens=300,
                      batch_size=BATCH_SIZE, seed=SEED)
    true = [ex["rating"] for ex in sample]
    pred = [P.parse_judge_output(g["text"])[0] for g in gens]
    metrics = E.judge_metrics(true, pred)
    metrics["model"] = MODEL
    (RESULTS / "judge_metrics.json").write_text(json.dumps(metrics, indent=1))
    (RESULTS / "judge_raw.jsonl").write_text("\n".join(
        json.dumps({"task_id": ex["task_id"], "true": t, "pred": p, "text": g["text"]})
        for ex, t, p, g in zip(sample, true, pred, gens)))
    summary["judge"] = metrics
    print(json.dumps(metrics, indent=1))
    print("\nExample output:\n", gens[0]["text"][:600])

QWK matters more than accuracy here: always guessing the most common rating gets about 33% accuracy but QWK 0.

In [ ]:
(RESULTS / "summary.json").write_text(json.dumps(summary, indent=1))
print(f"Saved to {RESULTS}")
for name, s in summary.items():
    if name == "judge":
        print(f"judge: parse {s['parse_rate']:.0%}  acc {s.get('accuracy', 0):.0%}  "
              f"QWK {s.get('qwk', float('nan')):.2f}  (majority baseline {s['majority_baseline_accuracy']:.0%})")
    else:
        print(f"{name}: pass@1+ {s['pass@1_plus']:.1%}  [{s['ci95'][0]:.1%}-{s['ci95'][1]:.1%}]")
if QUICK_CHECK:
    print("\nQUICK_CHECK was on. If the outputs look right, set QUICK_CHECK = False and run all cells again.")